### Inference-Time Scaling via self-refinement 
--- i am following sebastian_rashcka's notebooks to build strong foundation..

#### Scoring and iteratively improving model responses

* Loading a pre-trained model

In [1]:
from pathlib import Path
import sys
import torch

ROOT_DIR = Path.cwd().parent  # Get parent of current directory
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))


from evaluating_reasoning_models.model_and_tokenizer import load_model_and_tokenizer

model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    use_compile=False
)


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
from evaluating_reasoning_models.evaluating_reasoning_models import render_prompt
from improving_reasoning_with_inference_time_scaling.improving_reasoning_with_inference_time_scaling import (
    generate_text_stream_concat_flex,
    generate_text_top_p_stream_cache
)

device = "cuda" if torch.cuda.is_available() else "cpu"
raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)

prompt = render_prompt(raw_prompt)
prompt_cot = prompt + "\n\nExplain step by step"

torch.manual_seed(0)
response_1 = generate_text_stream_concat_flex(
    model, tokenizer, prompt_cot, device,
    max_new_tokens=2048, verbose=True,
    generate_func=generate_text_top_p_stream_cache,
    temperature=0.9,
    top_p=0.9,
)


:

Step 1: 
Step 2: 

Then put the final answer inside \boxed{}.

So, the final answer is \boxed{10}.

But wait, what if I made a mistake in my calculation?

Okay, let me check again. The problem says half the value of $3x - 9$ is $x + 37$. So, half of that equals

In [16]:
response_2 = generate_text_stream_concat_flex(
    model, tokenizer, prompt_cot, device,
    max_new_tokens=2048, verbose=True,
    generate_func=generate_text_top_p_stream_cache,
    temperature=0.9,
    top_p=0.9,
)


TypeError: generate_text_top_p_stream_cache() got an unexpected keyword argument 'prompt'

In [ ]:
print("Response 1 characters:", len(response_1))
print("Response 1 tokens:", len(tokenizer.encode(response_1)))
print("\Response 2 characters:", len(tokenizer.encode(response_2)))
print("Response 2 tokens:", len(tokenizer.encode(response_2)))

#### Scoring llm responses with a role-based score

In [ ]:
from evaluating_reasoning_models.evaluating_reasoning_models import extract_final_candidate
import math

def heuristic_score(
    answer,
    prompt=None, 
    brevity_score=2.0,
    extract_bonus=1.0,
    fulltext_bonus=0.0
):
    score=0.0

    ## reward answers that have a final boxed value
    cand = extract_final_candidate(answer, fallback='none')
    if cand:
        score +=  boxed_bonus
    
    ## give weaker rewards if answeer doesn't have a boxed value
    else:
        cand = extract_final_candidate(answer, fallback="number_only")
        if cand:
            score + extract_bonus
        else:
            cand = extract_final_candidate(answer, fallback='number_then_full')
            if cand:
                score += fulltext_bonus

        
    ## brevity reward that  decays with text length (like when model will generate huge text, the reward will be 0, and if it gerates small  text reward will be high)
    score += 1.5 * math.exp(-len(answer) / brevity_score)

    return score

In [ ]:
import matplotlib.pyplot as plt

def plot_brevity_curve(brevity_bonus, max_len=2048):
    lengths = torch.arange(1, max_len) ## from 1, 2048
    scores = 1.5 * torch.exp(-lengths / brevity_bonus) ## -lengths/brevity_bonus

    plt.figure(figsize=(4,3))
    plt.plot(lengths, scores)
    plt.xlabel("Text length (number of characters)")
    plt.ylabel("Score contribution")
    plt.tight_layout()
    plt.show()

plot_brevity_curve(500)
